<a href="https://colab.research.google.com/github/m-manuelmussa/Drug-Repurposing-IPEs-MtrD-Pharmacophores-Molecular-Docking-Dynamics/blob/main/1_Agrega%C3%A7%C3%A3o_de_medicamentos_aprovados_e_curadoria_b%C3%A1sica_ChEMBL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Agregação de medicamentos aprovados e curadoria básica**
**Objectivo**: Reunir e curar medicamentos aprovados no ChEMBL



***Autor: Micliete Lopes Manuel Mussa***

## **1. Instalação de Dependências**

In [1]:
!pip install chembl_webresource_client pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.8/70.8 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 5.9 MB/s eta 0:00:00


## **2. Importação de bibliotecas**

In [2]:
import pandas as pd
from chembl_webresource_client.new_client import new_client

/usr/local/lib/python3.13/dist-packages/chembl_webresource_client/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __version__ = __import__('pkg_resources').get_distribution('chembl_webresource_client').version


## **3. Busca de fármacos totais aprovados**

In [7]:
def buscar_medicamentos_aprovados():
    molecule = new_client.molecule

    # 1. Requisição adicionando os filtros: max_phase=4, Small molecule e estrutura MOL
    approved_drugs = molecule.filter(
        max_phase=4,
        molecule_type='Small molecule',
        structure_type='MOL'
    ).only(
        'molecule_chembl_id',
        'pref_name',
        'molecule_type',
        'first_approval',
        'structure_type',
        'molecule_structures'
    )

    print(f"Total de small molecules aprovadas encontradas: {len(approved_drugs)}")

    df = pd.DataFrame(list(approved_drugs))

    # 2. Extrai a string SMILES do dicionário interno
    df['smiles'] = df['molecule_structures'].apply(
        lambda x: x.get('canonical_smiles') if isinstance(x, dict) else None
    )

    # 3. Remove a coluna aninhada e renomeia as demais
    df = df.drop(columns=['molecule_structures']).rename(columns={
        'molecule_chembl_id': 'chembl_id',
        'pref_name': 'nome_preferencial',
        'molecule_type': 'tipo_molecula',
        'first_approval': 'ano_primeira_aprovacao',
        'structure_type': 'tipo_estrutura'
    })

    return df

# Executa a função e armazena no DataFrame
df_aprovados = buscar_medicamentos_aprovados()

Total de small molecules aprovadas encontradas: 3310


In [8]:
# Exibe as primeiras linhas
df_aprovados.head()

,ano_primeira_aprovacao,chembl_id,tipo_molecula,nome_preferencial,tipo_estrutura,smiles
0,1976.0,CHEMBL2,Small molecule,PRAZOSIN,MOL,COc1cc2nc(N3CCN(C(=O)c4ccco4)CC3)nc(N)c2cc1OC
1,1984.0,CHEMBL3,Small molecule,NICOTINE,MOL,CN1CCC[C@H]1c1cccnc1
2,1990.0,CHEMBL4,Small molecule,OFLOXACIN,MOL,CC1COc2c(N3CCN(C)CC3)c(F)cc3c(=O)c(C(=O)O)cn1c23
3,1964.0,CHEMBL5,Small molecule,NALIDIXIC ACID,MOL,CCn1cc(C(=O)O)c(=O)c2ccc(C)nc21
4,1965.0,CHEMBL6,Small molecule,INDOMETHACIN,MOL,COc1ccc2c(c1)c(CC(=O)O)c(C)n2C(=O)c1ccc(Cl)cc1


## **4. Curadoria Básica dos fármacos aprovados**


In [9]:
# 1. Remove registros com SMILES ausente/nulo (NaN/None)
df_curado = df_aprovados.dropna(subset=['smiles']).copy()

# 2. Remove registros com SMILES vazios ou contendo apenas espaços
df_curado = df_curado[df_curado['smiles'].str.strip() != ""]
print(f"Após remover fármacos sem SMILES: {len(df_curado)}")

# 3. Remove estruturas SMILES duplicadas (mantém a primeira ocorrência)
df_curado = df_curado.drop_duplicates(subset=['smiles'], keep='first')
print(f"Após remover SMILES duplicados: {len(df_curado)}")

# 4. Reorganiza o índice do DataFrame
df_curado = df_curado.reset_index(drop=True)

Após remover fármacos sem SMILES: 3310
Após remover SMILES duplicados: 3310


In [10]:
# Exibe a contagem final e as primeiras linhas
df_curado.head()

,ano_primeira_aprovacao,chembl_id,tipo_molecula,nome_preferencial,tipo_estrutura,smiles
0,1976.0,CHEMBL2,Small molecule,PRAZOSIN,MOL,COc1cc2nc(N3CCN(C(=O)c4ccco4)CC3)nc(N)c2cc1OC
1,1984.0,CHEMBL3,Small molecule,NICOTINE,MOL,CN1CCC[C@H]1c1cccnc1
2,1990.0,CHEMBL4,Small molecule,OFLOXACIN,MOL,CC1COc2c(N3CCN(C)CC3)c(F)cc3c(=O)c(C(=O)O)cn1c23
3,1964.0,CHEMBL5,Small molecule,NALIDIXIC ACID,MOL,CCn1cc(C(=O)O)c(=O)c2ccc(C)nc21
4,1965.0,CHEMBL6,Small molecule,INDOMETHACIN,MOL,COc1ccc2c(c1)c(CC(=O)O)c(C)n2C(=O)c1ccc(Cl)cc1


## **5. Exportação do Dataset final**

In [11]:
# Salva o DataFrame curado em um arquivo CSV
df_curado.to_csv("medicamentos_aprovados_chembl_curado.csv", index=False, encoding='utf-8')

print("Arquivo 'medicamentos_aprovados_chembl_curado.csv' salvo com sucesso!")

Arquivo 'medicamentos_aprovados_chembl_curado.csv' salvo com sucesso!


***Autor: Micliete Lopes Manuel Mussa, Farmacêutico.***